# Lab | Music Recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend songs based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find your own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrates how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [ ]:
!pip install -qU librosa panns-inference datasets pinecone-client==3.1.0 python-dotenv

# Download PANNs Model Data

`panns_inference` internally relies on `wget` to download a label CSV and the model checkpoint. On macOS (and systems without `wget`) this silently fails, causing a `FileNotFoundError` at import time. We solve this by downloading the required files ourselves with Python's built-in `urllib` before importing the library.

In [ ]:
import os
import urllib.request

# Create the directory that panns_inference expects
panns_dir = os.path.expanduser('~/panns_data')
os.makedirs(panns_dir, exist_ok=True)

# Download the AudioSet class labels CSV (needed by panns_inference at import time)
csv_path = os.path.join(panns_dir, 'class_labels_indices.csv')
if not os.path.exists(csv_path):
    print('Downloading class_labels_indices.csv ...')
    urllib.request.urlretrieve(
        'http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv',
        csv_path
    )
    print('Done.')
else:
    print('class_labels_indices.csv already present, skipping download.')

# Download the Cnn14 model checkpoint (needed by AudioTagging)
model_path = os.path.join(panns_dir, 'Cnn14_mAP=0.431.pth')
if not os.path.exists(model_path):
    print('Downloading Cnn14 model checkpoint (~300 MB) ...')
    urllib.request.urlretrieve(
        'https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth',
        model_path
    )
    print('Done.')
else:
    print('Model checkpoint already present, skipping download.')

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, each 5 seconds long. The dataset can be loaded from the HuggingFace model hub as follows:

In [ ]:
from datasets import load_dataset

# Load the ESC-50 dataset from the HuggingFace hub
data = load_dataset('ashraq/esc50', split='train')
data

The audios in the dataset are sampled at 44100 Hz and loaded into NumPy arrays. Let's take a look.

In [ ]:
# Display the first three audio entries
audios_raw = data['audio']
audios_raw[:3]

We only need the NumPy arrays as these contain all of the audio data. We will later feed these arrays directly into our embedding model to generate audio embeddings.

In [ ]:
import numpy as np

# Extract the raw audio arrays from the dataset and stack them into one NumPy array
audios = np.array([a['array'] for a in data['audio']])
print(f'Audio array shape: {audios.shape}')  # Expected: (2000, 220500)

# Load Audio Embedding Model

We use the *Cnn14* model from the *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate 2048-dimensional audio embeddings. The `panns_inference` package provides a convenient interface.

> **Note:** We pass `checkpoint_path=model_path` (the file we downloaded above) so the library does not try to use `wget`.

In [ ]:
from panns_inference import AudioTagging

# Load the Cnn14 model using the checkpoint we downloaded manually
# Use device='cuda' if a GPU is available, otherwise keep 'cpu'
model = AudioTagging(checkpoint_path=model_path, device='cpu')
print('Model loaded successfully.')

## Initializing the Pinecone Index

We need a place to store these embeddings and enable efficient vector search. Get a free API key at [app.pinecone.io](https://app.pinecone.io/) and add it to a `.env` file as `PINECONE_API_KEY=your_key_here`.

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os

# Load environment variables from a local .env file
load_dotenv(find_dotenv())

PINECONE_API_KEY = os.getenv('PINECONE_API_KEY') or 'YOUR_API_KEY'

if PINECONE_API_KEY == 'YOUR_API_KEY':
    print('WARNING: No Pinecone API key found. Set PINECONE_API_KEY in your .env file.')
else:
    print('Pinecone API key loaded.')

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import time
import os
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Define the cloud and region for the serverless index
cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'
spec = ServerlessSpec(cloud=cloud, region=region)

index_name = 'audio-search-demo'

# Create the index if it does not already exist
if index_name not in pc.list_indexes().names():
    print(f'Creating index "{index_name}" ...')
    pc.create_index(
        index_name,
        dimension=2048,  # Cnn14 embedding size
        metric='cosine',
        spec=spec
    )
    # Wait until the index is ready
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print('Index created.')
else:
    print(f'Index "{index_name}" already exists, connecting ...')

# Connect to the index and show its current stats
index = pc.Index(index_name)
index.describe_index_stats()

# Generate Embeddings and Upsert

Now we generate embeddings for all 2000 audio samples and upsert them into Pinecone. We do this in batches to stay within memory and API limits.

In [ ]:
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # Determine the end of this batch
    i_end = min(i + batch_size, len(audios))
    batch = audios[i:i_end]

    # Generate 2048-dimensional embeddings for the batch
    _, emb = model.inference(batch)

    # Create string IDs that match the dataset indices
    ids = [str(idx) for idx in range(i, i_end)]

    # Upsert (id, vector) pairs into Pinecone
    index.upsert(vectors=list(zip(ids, emb.tolist())))

# Confirm all vectors are indexed
index.describe_index_stats()

We now have *2000* audio records indexed in Pinecone — ready to query.

# Querying

Let's pick an audio sample from the dataset, listen to it, generate its embedding, and use it to find the most similar sounds in our Pinecone index.

In [ ]:
from IPython.display import Audio, display

# Choose a sample to use as the query
audio_num = 400
query_audio = data[audio_num]['audio']['array']
category = data[audio_num]['category']

print('Query Audio:', category)
Audio(query_audio, rate=44100)

We have the sound of a car horn. Let's generate an embedding for it.

In [ ]:
# panns_inference expects a 2D array: (num_samples, audio_length)
query_input = query_audio[None, :]

# Generate the embedding
_, xq = model.inference(query_input)
print(f'Query embedding shape: {xq.shape}')  # Expected: (1, 2048)

We have converted the audio into a 2048-dimensional vector. Let's use this to query our Pinecone index.

In [ ]:
# Query Pinecone with the embedding vector
results = index.query(vector=xq.tolist(), top_k=3)
results

The top result (id=400) is the query audio itself — the most similar item to any query is always itself. Let's listen to the top 3 results.

In [ ]:
# Play the top 3 most similar audio samples
for r in results['matches']:
    a = data[int(r['id'])]['audio']['array']
    print(f"id={r['id']}, score={r['score']:.4f}, category={data[int(r['id'])]['category']}")
    display(Audio(a, rate=44100))

Great results — everything aligns with a busy city street with car horns.

Let's write a helper function to easily run queries by dataset ID. Since the audio is already indexed, we can query Pinecone directly by vector ID without re-embedding.

In [ ]:
def find_similar_audios(audio_id, top_k=5):
    """Query Pinecone by dataset ID and play the most similar audio samples."""
    print(f'Query Audio (id={audio_id}, category={data[audio_id]["category"]}):')    
    query_audio = data[audio_id]['audio']['array']
    display(Audio(query_audio, rate=44100))

    # Use fetch+query instead of deprecated id= parameter
    query_input = query_audio[None, :]
    _, xq = model.inference(query_input)
    result = index.query(vector=xq.tolist(), top_k=top_k)

    print(f'Top {top_k} similar results:')
    for r in result['matches']:
        idx = int(r['id'])
        print(f"  id={r['id']}, score={r['score']:.4f}, category={data[idx]['category']}")
        display(Audio(data[idx]['audio']['array'], rate=44100))

In [ ]:
find_similar_audios(1642)

We get a set of revving motors — either vehicles or lawnmowers.

In [ ]:
find_similar_audios(452)

A relaxing set of birds chirping in nature.

Let's now try an audio sample from *outside* the dataset to see how the search generalises.

## Query with an External Audio File

We download a cat meow sample from Google's AudioSet using Python's `urllib` (replacing the original `wget` call that fails on macOS).

In [ ]:
import os
import urllib.request

# Download the external audio sample if not already present
external_audio_path = './miaow_16k.wav'
if not os.path.exists(external_audio_path):
    print('Downloading miaow_16k.wav ...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/audioset/miaow_16k.wav',
        external_audio_path
    )
    print('Done.')
else:
    print('File already present, skipping download.')

Load the downloaded audio file using `librosa` and listen to it:

In [ ]:
import librosa

# Load and resample the external audio to 44100 Hz (the sample rate used by our index)
a, _ = librosa.load(external_audio_path, sr=44100)
print(f'Loaded audio shape: {a.shape}')
Audio(a, rate=44100)

Now generate the embedding and query Pinecone:

In [ ]:
# Generate embedding for the external audio
query_input = a[None, :]
_, xq = model.inference(query_input)

# Query the index
results = index.query(vector=xq.tolist(), top_k=3)

# Play the top 3 similar audios
print('Top 3 similar audio samples from the index:')
for r in results['matches']:
    idx = int(r['id'])
    print(f"  id={r['id']}, score={r['score']:.4f}, category={data[idx]['category']}")
    display(Audio(data[idx]['audio']['array'], rate=44100))

The search correctly identifies similar cat sounds — excellent.

# Delete the Index

Delete the index when you no longer need it. **This action is irreversible** — the index and all its vectors will be permanently removed.

In [ ]:
# Uncomment the line below to permanently delete the index
pc.delete_index(index_name)